# 024 checks

# Imports

In [ ]:
import json
import os

os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from matplotlib.colors import TwoSlopeNorm
from matplotlib.figure import Figure
from tqdm.auto import tqdm

from uavloc.data.drone_frames import DroneFrame, frame_place, load_frames
from uavloc.data.match_pairs import PairBank, PairBankConfig, build_pair_bank, village_split
from uavloc.data.multidate import load_date
from uavloc.features.factory import correlator_config
from uavloc.features.segmentation import CLASSES, SegmentationFeatures, load_segmenter
from uavloc.geo import LatLon
from uavloc.matching.benchmark import benchmark_draws
from uavloc.matching.compatibility import aligned_joint, bank_joint, pmi_from_joint, register_pmi
from uavloc.matching.correlator import PositionCorrelator, z_scored
from uavloc.matching.identity import SatelliteAsDroneCorrelator
from uavloc.track import REPO_ROOT, experiment_dir, load_runs, set_seed

# Params

In [ ]:
EXP = "024-class-compatibility-matching"
DEVICE = "mps"
SEED = 42
NUM_THREADS = 2

PLACES = ["bushcha-50p2422-25p8305", "birky-49p9105-23p9192"]
FRAMES_DATASET = "village-crossdate-frames"
MAP_DATE = "newest"
CLASS_WEIGHTS = (0.0, 1.0, 1.0, 0.5, 0.0, 0.5)
SEARCH_SCALE = 1.7
FOV_W_DEG = 60.0
SAT_FEATURE_CROP_PCT = 0.03
SEG_ARCHITECTURE = "lraspp"
SEG_CHECKPOINT = "pocs/mobilenet-lraspp-speed/model-lraspp-bn-satellite.pt"
SEG_CHANNELS = "all"
MATCH_GSD_M = 0.385

IDENTITY_FRAMES = 5
IDENTITY_NOISE_M = 10.0
N_DIRECTIONS = 8
MAX_CELL_ERROR = 1.0              # argmax within one cell of the truth, per axis
MI_FLOOR = 1e-8

EXCLUDE_PLACES = ["birky-49p9105-23p9192", "bushcha-50p2422-25p8305", "dubrova-49p5175-24p0765"]
N_VAL_VILLAGES = 10
SPLIT_SEED = 42
DRONE_DATE = "current"
ALTITUDE_M = 40.0
MARGIN_M = 80.0
N_TRAIN_PAIRS = 20_000
BANK_SEED = 42
PMI_SMOOTHING = 1e-4
SAME_DATE_PAIRS = 1_000
MIN_CLASS_SHARE = 1e-3            # a class with less soft mass than this on either side is reported, not judged

BASELINE_EXP = "007-search-prior-and-gate"
BASELINE_RUN = "007.1"
CONTROL_RUN = "024.1"
REPRODUCE_TOL_M = 0.001

# Data

In [ ]:
set_seed(SEED)
torch.set_num_threads(NUM_THREADS)
checks = {}

In [ ]:
train_villages, val_villages = village_split(EXCLUDE_PLACES, N_VAL_VILLAGES, SPLIT_SEED)
checks["split"] = {"train": len(train_villages), "val": len(val_villages),
                   "excluded_absent": not set(EXCLUDE_PLACES) & set(train_villages + val_villages)}
checks["split"]

In [ ]:
frames = [f for f in load_frames(FRAMES_DATASET) if frame_place(f) in PLACES]
identity_frames = [frames[i] for i in np.linspace(0, len(frames) - 1, IDENTITY_FRAMES).round().astype(int)]
[(f.name, frame_place(f)) for f in identity_frames]

# Model

In [ ]:
model = load_segmenter(REPO_ROOT / SEG_CHECKPOINT, DEVICE, SEG_ARCHITECTURE)
features = {place: SegmentationFeatures(model, model, load_date(place, MAP_DATE), SEG_CHANNELS, MATCH_GSD_M,
                                        MATCH_GSD_M, DEVICE) for place in PLACES}

In [ ]:
PAIR_CONFIG = PairBankConfig(drone_date=DRONE_DATE, altitude_m=ALTITUDE_M, fov_w_deg=FOV_W_DEG, margin_m=MARGIN_M,
                             search_scale=SEARCH_SCALE, match_gsd_m=MATCH_GSD_M,
                             sat_feature_crop_pct=SAT_FEATURE_CROP_PCT, seg_channels=SEG_CHANNELS,
                             seg_checkpoint=SEG_CHECKPOINT)
bank = build_pair_bank(train_villages, N_TRAIN_PAIRS, PAIR_CONFIG, model, DEVICE, BANK_SEED)
bank.root.name, len(bank)

In [ ]:
joint_all = bank_joint(bank)
W = pmi_from_joint(joint_all, PMI_SMOOTHING)
register_pmi(W)
pd.DataFrame(W, index=CLASSES, columns=CLASSES).round(2)

# Identity

In [ ]:
def bench_config(matching: str):
    """007.1's geometry with `matching`; hand weights only for weighted-ncc."""
    return correlator_config({"search_scale": SEARCH_SCALE, "fov_w_deg": FOV_W_DEG,
                              "sat_feature_crop_pct": SAT_FEATURE_CROP_PCT, "drone_feature_crop_pct": 0.0,
                              "matching": matching,
                              "class_weights": CLASS_WEIGHTS if matching == "weighted-ncc" else None})

In [ ]:
def identity_rows(matching: str) -> list[dict]:
    """Satellite crop at the true pose as the query: argmax cell vs true cell, per frame x direction."""
    rows = []
    for frame in tqdm(identity_frames, desc=matching):
        correlator = SatelliteAsDroneCorrelator(features[frame_place(frame)], bench_config(matching))
        for draw in benchmark_draws(correlator, [frame], IDENTITY_NOISE_M, N_DIRECTIONS, SEED):
            surface = draw.surface
            true_cell = surface.cell_of_offset(PositionCorrelator.patch_offset_m(draw.prior, frame.position, frame.yaw_deg))
            best = np.unravel_index(np.nanargmax(surface.heatmap), surface.heatmap.shape)
            rows.append({"matching": matching, "frame": frame.name, "bearing_deg": draw.bearing_deg,
                         "reachable": draw.reachable,
                         "flat": bool(np.nanmax(surface.heatmap) == np.nanmin(surface.heatmap)),
                         "cell_error": max(abs(best[0] - true_cell[0]), abs(best[1] - true_cell[1]))})
    return rows

In [ ]:
identity = pd.DataFrame([row for matching in ("pmi", "mi", "weighted-ncc") for row in identity_rows(matching)])
identity.groupby(["matching", "reachable"])[["cell_error", "flat"]].agg(["mean", "max", "count"])

In [ ]:
def identity_summary(group: pd.DataFrame) -> dict:
    """Pass: every reachable draw has its argmax within one cell; an unreachable truth lies outside the surface."""
    reachable = group[group["reachable"]]
    return {"draws": len(group), "reachable_draws": len(reachable), "flat": int(group["flat"].sum()),
            "within_one_cell_share_all": float((group["cell_error"] <= MAX_CELL_ERROR).mean()),
            "within_one_cell_share_reachable": float((reachable["cell_error"] <= MAX_CELL_ERROR).mean()),
            "max_cell_error_reachable": float(reachable["cell_error"].max()),
            "pass": bool((reachable["cell_error"] <= MAX_CELL_ERROR).all())}

In [ ]:
checks["identity"] = {matching: identity_summary(group) for matching, group in identity.groupby("matching")}
checks["identity"]

# Identity parity

In [ ]:
def direct_score(drone: torch.Tensor, satellite: torch.Tensor, matching: str) -> np.ndarray:
    """Raw PMI or MI score from 36 direct float64 conv2d calls: the reference for the FFT path."""
    drone, satellite = drone.cpu().double(), satellite.cpu().double()
    n = drone[0, 0].numel()
    joint = torch.stack([torch.stack([F.conv2d(satellite[:, j:j + 1], drone[:, i:i + 1])[0, 0] / n
                                      for j in range(len(CLASSES))]) for i in range(len(CLASSES))])
    if matching == "pmi":
        return (torch.as_tensor(W)[:, :, None, None] * joint).sum((0, 1)).numpy()
    joint = joint.clamp(min=MI_FLOOR)
    log_ratio = joint.log() - joint.sum(1, keepdim=True).clamp(min=MI_FLOOR).log() - joint.sum(0, keepdim=True).clamp(min=MI_FLOOR).log()
    return (joint * log_ratio).sum((0, 1)).numpy()

In [ ]:
def parity_row(row: pd.Series) -> dict:
    """One identity draw redone directly: FFT-vs-direct z difference, query vs map at the truth, score there vs max."""
    frame = next(f for f in identity_frames if f.name == row["frame"])
    correlator = SatelliteAsDroneCorrelator(features[frame_place(frame)], bench_config(row["matching"]))
    draw = next(d for d in benchmark_draws(correlator, [frame], IDENTITY_NOISE_M, N_DIRECTIONS, SEED)
                if np.isclose(d.bearing_deg, row["bearing_deg"]))
    drone, sat, _, _ = correlator.prepare(correlator.drone_map(frame), draw.prior, frame.yaw_deg, correlator.patch_size_m(frame))
    raw = direct_score(drone, sat, row["matching"])
    cell = draw.surface.cell_of_offset(PositionCorrelator.patch_offset_m(draw.prior, frame.position, frame.yaw_deg))
    r, c = int(round(cell[0])), int(round(cell[1]))
    window = sat[0, :, r:r + drone.shape[2], c:c + drone.shape[3]].cpu()
    return {**row.to_dict(), "max_z_diff_fft_vs_direct": float(np.abs(z_scored(torch.from_numpy(raw)) - draw.surface.heatmap).max()),
            "query_vs_map_at_truth": float((window - drone[0].cpu()).abs().mean()),
            "score_at_truth": float(raw[r, c]), "score_max": float(raw.max()),
            "z_at_truth": float(draw.surface.heatmap[r, c]), "z_max": float(np.nanmax(draw.surface.heatmap))}

In [ ]:
missed = identity[identity["reachable"] & (identity["cell_error"] > MAX_CELL_ERROR) & identity["matching"].isin(["pmi", "mi"])]
parity = pd.DataFrame([parity_row(row) for _, row in tqdm(missed.iterrows(), total=len(missed), desc="parity")])
checks["identity_parity"] = {"draws": len(parity),
                             "max_z_diff_fft_vs_direct": float(parity["max_z_diff_fft_vs_direct"].max()),
                             "max_query_vs_map_at_truth": float(parity["query_vs_map_at_truth"].max()),
                             "median_z_gap_truth_to_max": parity.groupby("matching").apply(
                                 lambda g: float((g["z_max"] - g["z_at_truth"]).median()), include_groups=False).to_dict()}
parity.round(4)

# Same-date W

In [ ]:
def same_date_joint(indices: np.ndarray) -> np.ndarray:
    """`aligned_joint` over bank pairs with the map re-cut from the query's own date at the same prior."""
    pairs = bank.pairs.iloc[indices]
    joint = None
    for place, rows in tqdm(pairs.groupby("place"), desc="same-date places"):
        place_features = SegmentationFeatures(model, model, load_date(place, DRONE_DATE), SEG_CHANNELS,
                                              MATCH_GSD_M, MATCH_GSD_M, DEVICE)
        correlator = PositionCorrelator(place_features, PAIR_CONFIG.correlator_config())
        batch = bank.batch(rows.index.to_numpy())
        satellites = [correlator.prepare(query[None], LatLon(row.prior_lat, row.prior_lon), row.yaw_deg,
                                         PAIR_CONFIG.patch_size_m)[1][0].cpu()
                      for query, row in zip(batch["query"], rows.itertuples())]
        part = aligned_joint(batch["query"], torch.stack(satellites), batch["cell"])
        joint = part if joint is None else joint + part
    return joint.numpy()

In [ ]:
same_indices = np.sort(np.random.default_rng(SEED).choice(len(bank), SAME_DATE_PAIRS, replace=False))
joint_same = same_date_joint(same_indices)
joint_cross = bank_joint(bank, indices=same_indices)
W_same = pmi_from_joint(joint_same, PMI_SMOOTHING)
W_cross = pmi_from_joint(joint_cross, PMI_SMOOTHING)
pd.concat({"same-date": pd.DataFrame(W_same, index=CLASSES, columns=CLASSES),
           "cross-date": pd.DataFrame(W_cross, index=CLASSES, columns=CLASSES)}).round(2)

In [ ]:
def diagonal_margins(weights: np.ndarray) -> np.ndarray:
    """Per class: W_ii minus the largest off-diagonal entry in its row and column (> 0: diagonal dominates)."""
    off = np.where(np.eye(len(weights), dtype=bool), -np.inf, weights)
    return np.diag(weights) - np.maximum(off.max(1), off.max(0))

In [ ]:
def marginals(joint: np.ndarray) -> dict[str, dict[str, float]]:
    """Soft class shares of the drone (rows) and satellite (columns) side of a joint count."""
    p = joint / joint.sum()
    return {"drone": dict(zip(CLASSES, p.sum(1).round(6).tolist())), "satellite": dict(zip(CLASSES, p.sum(0).round(6).tolist()))}

In [ ]:
def supported(joint: np.ndarray) -> np.ndarray:
    """Classes with at least `MIN_CLASS_SHARE` of the soft mass on both sides."""
    p = joint / joint.sum()
    return (p.sum(1) >= MIN_CLASS_SHARE) & (p.sum(0) >= MIN_CLASS_SHARE)

In [ ]:
checks["same_date_diagonal"] = {
    "pairs": SAME_DATE_PAIRS,
    "margin_same": dict(zip(CLASSES, diagonal_margins(W_same).round(3).tolist())),
    "margin_cross": dict(zip(CLASSES, diagonal_margins(W_cross).round(3).tolist())),
    "diag_same": dict(zip(CLASSES, np.diag(W_same).round(3).tolist())),
    "W_same": W_same.round(4).tolist(), "W_cross": W_cross.round(4).tolist(),
    "pass": bool((diagonal_margins(W_same) > 0).all() and (np.diag(W_same) > 0).all()),
    "supported": dict(zip(CLASSES, supported(joint_same).tolist())),
    "pass_supported": bool(((diagonal_margins(W_same) > 0) & (np.diag(W_same) > 0))[supported(joint_same)].all()),
    "marginals_same": marginals(joint_same), "marginals_024_2": marginals(joint_all),
}
checks["same_date_diagonal"]

# Reproduction

In [ ]:
def run_metric(exp: str, run_id: str, key: str = "mean_position_error_m") -> float:
    return next(r for r in load_runs(exp) if r["run"] == run_id)["metrics"][key]

In [ ]:
control = pd.read_csv(experiment_dir(EXP) / "artifacts" / f"records-{CONTROL_RUN}.csv")
baseline = pd.read_csv(experiment_dir(BASELINE_EXP) / "artifacts" / f"records-{BASELINE_RUN}.csv")
keys = ["frame", "noise_m", "bearing_deg"]
paired = control.merge(baseline, on=keys, suffixes=("", "_baseline"))
checks["reproduction"] = {
    "control_m": run_metric(EXP, CONTROL_RUN), "baseline_m": run_metric(BASELINE_EXP, BASELINE_RUN),
    "draws_matched": len(paired), "draws": len(control),
    "max_draw_error_diff_m": float((paired["error_m"] - paired["error_m_baseline"]).abs().max()),
}
checks["reproduction"]["diff_m"] = checks["reproduction"]["control_m"] - checks["reproduction"]["baseline_m"]
checks["reproduction"]["pass"] = bool(abs(checks["reproduction"]["diff_m"]) <= REPRODUCE_TOL_M)
checks["reproduction"]

# Plots

In [ ]:
def plot_tables(tables: dict[str, np.ndarray]) -> Figure:
    """PMI tables side by side on one diverging scale around 0, drone class in rows."""
    limit = float(max(np.abs(t).max() for t in tables.values()))
    fig, axes = plt.subplots(1, len(tables), figsize=(5.5 * len(tables), 4.5))
    for ax, (title, weights) in zip(axes, tables.items()):
        image = ax.imshow(weights, cmap="RdBu_r", norm=TwoSlopeNorm(0.0, -limit, limit))
        for (row, col), value in np.ndenumerate(weights):
            ax.text(col, row, f"{value:.2f}", ha="center", va="center", fontsize=8,
                    color="white" if abs(value) > limit * 0.6 else "black")
        ax.set_xticks(range(len(CLASSES)), CLASSES, rotation=45, ha="right"); ax.set_yticks(range(len(CLASSES)), CLASSES)
        ax.set_xlabel("satellite class j"); ax.set_ylabel("drone class i"); ax.set_title(title)
    fig.colorbar(image, ax=axes, label="log P(i,j) / P(i)P(j)")
    return fig

In [ ]:
fig_tables = plot_tables({f"same date, {SAME_DATE_PAIRS} pairs": W_same, f"cross date, same {SAME_DATE_PAIRS} pairs": W_cross,
                          f"cross date, all {len(bank)} pairs (024.2)": W})

# Save

In [ ]:
figures_dir = experiment_dir(EXP) / "figures"
figures_dir.mkdir(exist_ok=True)
fig_tables.savefig(figures_dir / "checks-pmi-same-vs-cross-date.png", dpi=150, bbox_inches="tight")
identity.to_csv(experiment_dir(EXP) / "artifacts" / "checks-identity.csv", index=False)
parity.to_csv(experiment_dir(EXP) / "artifacts" / "checks-identity-parity.csv", index=False)
(experiment_dir(EXP) / "artifacts" / "checks.json").write_text(json.dumps(checks, indent=1, default=float))
{name: check.get("pass", {k: v["pass"] for k, v in check.items() if isinstance(v, dict) and "pass" in v})
 for name, check in checks.items()}